In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

In [5]:
# %%
import datetime
import logging
import os

import pandas as pd
# /venv/lib/python3.12/site-packages/gspread_pandas/spread.py:401: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)` .replace("", np.nan)
pd.set_option('future.no_silent_downcasting', True)

import helpers.hdbg as hdbg
import helpers.henv as henv
import helpers.hio as hio
import helpers.hpandas as hpandas
import helpers.hprint as hprint
import helpers.hcache as hcache

#hcache.get_global_cache_info()
#hcache.clear_global_cache("all")

import config_root.config as cconfig

# %%
hdbg.init_logger(verbosity=logging.INFO)

_LOG = logging.getLogger(__name__)

_LOG.info("%s", henv.get_system_signature()[0])

hprint.config_notebook()

INFO  # Git
  branch_name='CmampTask11020_Compute_yamm_stats'
  hash='d2293f44a'
  # Last commits:
    * d2293f44a GP Saggese Update                                                            (    4 days ago) Wed Jan 22 00:04:59 2025  (HEAD -> CmampTask11020_Compute_yamm_stats, origin/CmampTask11020_Compute_yamm_stats)
    * 5404856e0 GP Saggese Update                                                            (    4 days ago) Tue Jan 21 19:42:57 2025           
    * 9685fb48f GP Saggese Lint                                                              (    6 days ago) Sun Jan 19 23:49:38 2025           
# Machine info
  system=Linux
  node name=9b21dea71c71
  release=6.10.14-linuxkit
  version=#1 SMP Fri Nov 29 17:22:03 UTC 2024
  machine=aarch64
  processor=aarch64
  cpu count=8
  cpu freq=None
  memory=svmem(total=8218251264, available=5728026624, percent=30.3, used=2271641600, free=701063168, active=2949562368, inactive=3359866880, buffers=1318604800, cached=3926941696, shared=107

In [3]:
import gspread
print(gspread.__version__)

import gspread_pandas
print(gspread_pandas.__version__)

#gspread_pandas.conf.get_config()
print(gspread_pandas.conf.get_config()["project_id"])

5.12.4
3.3.0
gspread-gp


In [24]:
lines = hio.from_file("../../family_offices.txt")
lines = lines.split("\n")

# Ensure data is divisible by 6 for clean processing
# if len(lines) % 7 != 0:
#     raise ValueError("The data does not align properly into rows of 6.")

# Split into groups of 6
data = [lines[i:i+6] for i in range(0, len(lines), 6)]

# Create DataFrame
df = pd.DataFrame(data, columns=['Family Office', 'Name', 'Email', 'City', 'State', 'Website'])

df

,Family Office,Name,Email,City,State,Website
0,Family Office,Name,Email,City,State,Website
1,Fleming Family & Partners Limited,Ahmet Feridun,ahmet.feridun@ffandp.com,London,–,-
2,Stamos Capital,Ron Marryott,rmarryott@stamoscapital.com,–,CA,https://www.stamoscapital.com/about/
3,TAG Associates,John Pantowich,jpantowich@tagassoc.com,New York,NY,–
4,HOLBEIN PARTNERS LLP,ANDERE RODGER,andere.rodger@holbeinpartners.com,London,–,http://www.holbeinpartners.com/
5,ICONIQ Capital,Vid Mahansaria,vid@iconiqcapital.com,San Francisco,CA,http://www.iconiqcapital.com/
6,Vedra Partners,JOACHIM GOTTSCHALK,joachim.gottschalk@vedrapartners.com,Lausanne,–,http://www.vedrapartners.com/
7,Ash Crest Corp,Tony Brita,tony@ashcrest.com,Fort Wayne,IN,–
8,Flynn Family Office,Evan Jehle,ejehle@ffollc.com,–,–,flynnfamilyoffice.com
9,Keep Bermuda Beautiful (KBB),Susan Harvey,smh@ibl.bm,Hamilton,Hamilton,http://www.kbb.bm


In [16]:
len(lines) % 7

4

In [12]:
df

,Family Office,Name,Email,City,State,Website
0,Family Office,Name,Email,City,State,Website
1,Fleming Family & Partners Limited,Open,Ahmet Feridun,ahmet.feridun@ffandp.com,London,–
2,–,Stamos Capital,Ron Marryott,rmarryott@stamoscapital.com,–,CA
3,https://www.stamoscapital.com/about/,TAG Associates,John Pantowich,jpantowich@tagassoc.com,New York,NY
4,–,HOLBEIN PARTNERS LLP,ANDERE RODGER,andere.rodger@holbeinpartners.com,London,–
5,http://www.holbeinpartners.com/,ICONIQ Capital,Vid Mahansaria,vid@iconiqcapital.com,San Francisco,CA
6,http://www.iconiqcapital.com/,Vedra Partners,JOACHIM GOTTSCHALK,joachim.gottschalk@vedrapartners.com,Lausanne,–
7,http://www.vedrapartners.com/,Ash Crest Corp,Tony Brita,tony@ashcrest.com,Fort Wayne,IN
8,–,Flynn Family Office,Evan Jehle,ejehle@ffollc.com,–,–
9,flynnfamilyoffice.com,Keep Bermuda Beautiful (KBB),Susan Harvey,smh@ibl.bm,Hamilton,Hamilton
